In [39]:
from __future__ import annotations

import time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from io import StringIO
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def _log(msg: str, *, enable: bool = True):
    if enable:
        print(f"{msg}")

# --- Constants (target pages) ---
INJURIES_URL = "https://www.transfermarkt.us/premier-league/verletztespieler/wettbewerb/GB1"
BALANCE_URL  = "https://www.transfermarkt.us/premier-league/transferbilanztabellenplatz/wettbewerb/GB1"


def _build_session() -> requests.Session:
    """
    Create a requests Session with:
    - Realistic headers (UA, Accept, Language, Referer)
    - Retry policy for transient errors (403, 429, 5xx)
    """
    s = requests.Session()
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.transfermarkt.us/",
        "Connection": "keep-alive",
    })
    retries = Retry(
        total=5,
        backoff_factor=0.6,
        status_forcelist=[403, 429, 500, 502, 503, 504],
        allowed_methods=["GET", "HEAD"]
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.mount("http://", HTTPAdapter(max_retries=retries))
    return s


def _fetch_html(url: str) -> str:
    """
    Try to fetch HTML via requests first.
    If blocked (e.g., 403) or no content, fall back to Selenium to render the page.
    """
    sess = _build_session()
    resp = sess.get(url, timeout=25)
    if resp.status_code == 200 and resp.text.strip():
        return resp.text

    # --- Fallback: Selenium (handles basic JS-rendered or cookie-banner pages) ---
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    try:
        driver.get(url)
        time.sleep(2.0)  # Simple wait; replace with WebDriverWait if needed
        return driver.page_source
    finally:
        driver.quit()


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize DataFrame columns:
    - Flatten MultiIndex columns if present
    - Lowercase + replace spaces with underscores
    """
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([c for c in map(str, col) if c and not str(c).startswith("Unnamed")]).strip("_")
            for col in df.columns
        ]
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    return df

def _read_first_table(html: str) -> pd.DataFrame:
    """
    Read the first HTML table using StringIO to avoid FutureWarning in pandas>=2.2.
    Normalize MultiIndex headers.
    """
    dfs = pd.read_html(StringIO(html), flavor="lxml")
    if not dfs:
        raise ValueError("No tables found on page")
    df = dfs[0]
    return _normalize_columns(df)


POS_KEYWORDS = {
    # common positions on Transfermarkt
    "Goalkeeper","Right-Back","Left-Back","Centre-Back",
    "Defensive Midfield","Central Midfield","Attacking Midfield",
    "Right Winger","Left Winger","Centre-Forward","Second Striker",
    "Wing-Back","Sweeper"
}

def _clean_injuries_raw(df: pd.DataFrame) -> pd.DataFrame:
    """
    The raw injuries table often splits one logical record across multiple rows:
    - Row A: "Player Position" ends up in 'player/position'; club/injury/until/value in correct cols
    - Row B/C: the player's name and the position sometimes spill into 'club'
    This function compacts those fragments into a single tidy row per player.
    """
    # unify expected columns
    cols = [c.lower() for c in df.columns]
    rename_map = {}
    for c in df.columns:
        lc = c.lower()
        if "player" in lc and "position" in lc: rename_map[c] = "player_position"
        elif lc.strip() == "club": rename_map[c] = "club"
        elif "injury" in lc: rename_map[c] = "injury"
        elif lc.strip() == "until": rename_map[c] = "until"
        elif "market" in lc and "value" in lc: rename_map[c] = "market_value"
    df = df.rename(columns=rename_map)

    # guard for missing columns
    for need in ["player_position","club","injury","until","market_value"]:
        if need not in df.columns:
            df[need] = pd.NA

    # mark record starts: whenever 'player_position' is not null
    starts = df["player_position"].notna().astype(int).cumsum()
    df["_rid"] = starts

    records = []
    for rid, g in df.groupby("_rid", sort=False):
        # candidate strings scattered in columns
        first = g.iloc[0]
        # 1) extract player & position
        #    - preferred: split the first cell into name + position using group evidence
        pp = str(first.get("player_position", "")).strip()
        # gather potential fragments from the group (some show up under 'club' column)
        fragments = set(str(x).strip() for x in g["club"].dropna().tolist())
        # infer position
        pos = None
        for x in fragments | {pp}:
            if any(x == k or k in x for k in POS_KEYWORDS):
                pos = next((k for k in POS_KEYWORDS if (x == k or k in x)), None)
                if pos:
                    break
        # infer name: remove position token from pp if present; otherwise take the longest fragment that is not a position and not currency
        name = pp
        if pos and pos in name:
            name = name.replace(pos, "").strip()
        if not name or name.lower() in {"nan", "none"}:
            cand = [x for x in fragments if x not in POS_KEYWORDS and not x.startswith("€")]
            if cand:
                name = max(cand, key=len)

        # 2) extract club (avoid position/name fragments)
        club_candidates = [x for x in fragments if x not in {name} and x not in POS_KEYWORDS and not x.startswith("€")]
        club = club_candidates[0] if club_candidates else pd.NA

        # 3) other fields from the first row
        injury = first.get("injury", pd.NA)
        until = first.get("until", pd.NA)
        value = first.get("market_value", pd.NA)

        records.append({
            "player": name,
            "position": pos,
            "club": club,
            "injury": injury,
            "until": until,
            "market_value": value
        })

    tidy = pd.DataFrame.from_records(records)
    # keep original market_value text as-is (no numeric parsing)
    # final column order
    tidy = tidy[["player","position","club","injury","until","market_value"]]
    return tidy

def _clean_balance_raw(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the 'transfer balance vs table position' table for the Premier League page.

    Assumes a raw table with columns similar to:
    ['transfer_position', 'wappen', 'club', 'club.1',
     'transfer_activity', 'league_position', 'difference']

    Produces a tidy frame with:
    ['transfer_rank', 'club', 'transfer_balance', 'league_position', 'difference']
    """
    df = df.copy()

    # 1) drop obvious badge/icon columns
    drop_like = [c for c in df.columns if "wappen" in str(c).lower() or "badge" in str(c).lower()]
    df = df.drop(columns=drop_like, errors="ignore")

    # 2) map known columns based on the Transfermarkt PL table
    rename = {}
    for c in df.columns:
        lc = str(c).strip().lower()
        if "transfer" in lc and "position" in lc:
            # e.g. "Transfer position"
            rename[c] = "transfer_rank"
        elif lc == "club":
            # main club name
            rename[c] = "club"
        elif lc == "club.1":
            # this column holds the euro string (net spend)
            rename[c] = "transfer_balance"
        elif "table" in lc or "league_position" in lc or "tabellenplatz" in lc:
            rename[c] = "league_position"
        elif "difference" in lc or "differenz" in lc:
            rename[c] = "difference"

    df = df.rename(columns=rename)

    # 3) build tidy frame with only the columns we care about
    cols_order = ["transfer_rank", "club", "transfer_balance", "league_position", "difference"]
    keep = [c for c in cols_order if c in df.columns]
    tidy = df[keep].copy()

    # 4) basic type cleaning
    if "transfer_balance" in tidy.columns:
        tidy["transfer_balance"] = tidy["transfer_balance"].astype(str)

    if "transfer_rank" in tidy.columns:
        tidy["transfer_rank"] = pd.to_numeric(tidy["transfer_rank"], errors="coerce")
    if "league_position" in tidy.columns:
        tidy["league_position"] = pd.to_numeric(tidy["league_position"], errors="coerce")
    if "difference" in tidy.columns:
        tidy["difference"] = pd.to_numeric(tidy["difference"], errors="coerce")

    if "league_position" in tidy.columns:
        lp = tidy["league_position"]
        if (("difference" not in tidy.columns) or tidy["difference"].isna().all()) and \
           lp.notna().any() and lp.min() <= 0:
            tidy["difference"] = lp
            tidy["league_position"] = tidy["transfer_rank"] - tidy["difference"]

    for c in ["transfer_rank", "league_position", "difference"]:
        if c in tidy.columns:
            tidy[c] = pd.to_numeric(tidy[c], errors="coerce")

    ordered = [c for c in ["transfer_rank", "club", "transfer_balance", "league_position", "difference"]
               if c in tidy.columns]
    return tidy[ordered]

def _parse_injuries_clubs_from_html(html: str, expected_len: int) -> list:
    """
    Extract club names from the injuries table by reading either:
    - <img alt="Club Name"> inside the club column, or
    - <a title="Club Name"> fallback
    Returns a list of length <= expected_len; remaining rows will be filled with <NA>.
    """
    soup = BeautifulSoup(html, "lxml")
    table = soup.find("table")
    clubs = []
    if not table:
        return clubs
    tbody = table.find("tbody") or table
    for tr in tbody.find_all("tr", recursive=False):
        tds = tr.find_all("td", recursive=False)
        if len(tds) < 2:
            continue
        club_td = tds[1]
        club = None
        img = club_td.find("img", alt=True)
        if img and img.get("alt"):
            club = img["alt"].strip()
        if not club:
            a = club_td.find("a", title=True)
            if a and a.get("title"):
                club = a["title"].strip()
        clubs.append(club if club else None)
        if len(clubs) >= expected_len:
            break
    return clubs

def _fill_injuries_club_with_html(injuries_df: pd.DataFrame, injuries_html: str) -> pd.DataFrame:
    """
    Fill missing 'club' values in injuries_df using parsed club names from the original HTML.
    """
    clubs = _parse_injuries_clubs_from_html(injuries_html, len(injuries_df))
    if clubs:
        injuries_df = injuries_df.copy()
        injuries_df["club"] = pd.Series(clubs, index=injuries_df.index)
    return injuries_df

def scrape_transfermarkt(verbose: bool = True):
    _log("Kicking off Transfermarkt run…", enable=verbose)

    # --- injuries ---
    _log("Fetching injuries page…", enable=verbose)
    injuries_html = _fetch_html(INJURIES_URL)
    _log("Parsing injuries table…", enable=verbose)
    injuries_raw  = _read_first_table(injuries_html)
    _log(f"Injuries raw rows: {len(injuries_raw)} — cleaning…", enable=verbose)
    injuries_df   = _clean_injuries_raw(injuries_raw)
    _log("Filling club names from logo/title…", enable=verbose)
    injuries_df   = _fill_injuries_club_with_html(injuries_df, injuries_html)
    _log(f"Injuries ready: {len(injuries_df)} rows.", enable=verbose)

    # --- balance vs position ---
    _log("Fetching balance-vs-position page…", enable=verbose)
    balance_html = _fetch_html(BALANCE_URL)
    _log("Parsing balance table…", enable=verbose)
    balance_raw  = _read_first_table(balance_html)
    _log(f"Balance raw rows: {len(balance_raw)} — cleaning…", enable=verbose)
    balance_df   = _clean_balance_raw(balance_raw)
    
    injuries_df = injuries_df.copy()
    injuries_df.index = range(1, len(injuries_df) + 1)
    balance_df = balance_df.copy()
    balance_df.index = range(1, len(balance_df) + 1)

    _log(f"Balance ready: {len(balance_df)} rows.", enable=verbose)
    _log("All done!", enable=verbose)

    return {"injuries": injuries_df, "balance_vs_position": balance_df}

In [40]:
out = scrape_transfermarkt(verbose=True)
injuries = out["injuries"]
balance_vs_position = out["balance_vs_position"]

Kicking off Transfermarkt run…
Fetching injuries page…
Parsing injuries table…
Injuries raw rows: 246 — cleaning…
Filling club names from logo/title…
Injuries ready: 82 rows.
Fetching balance-vs-position page…
Parsing balance table…
Balance raw rows: 20 — cleaning…
Balance ready: 20 rows.
All done!


In [41]:
injuries

,player,position,club,injury,until,market_value
1,Antonee Robinson,Left-Back,Fulham FC,Knee problems,NaN,€35.00m
2,Douglas Luiz,Central Midfield,Nottingham Forest,Thigh problems,NaN,€30.00m
3,Nick Pope,Goalkeeper,Newcastle United,Head injury,NaN,€8.00m
4,Joelinton,Central Midfield,Newcastle United,Shin injury,NaN,€35.00m
5,Harrison Ashby,Right-Back,Newcastle United,Thigh problems,NaN,€1.40m
...,...,...,...,...,...,...
78,Zeki Amdouni,Centre-Forward,Burnley FC,Cruciate ligament tear,"Apr 1, 2026",€10.00m
79,James Maddison,Attacking Midfield,Tottenham Hotspur,Cruciate ligament tear,"Jun 1, 2026",€35.00m
80,Levi Colwill,Centre-Back,Chelsea FC,Cruciate ligament tear,"Jun 1, 2026",€55.00m
81,Adam Webster,Centre-Back,Brighton & Hove Albion,Cruciate ligament injury,"Jun 1, 2026",€9.00m


In [42]:
balance_vs_position

,transfer_rank,club,transfer_balance,league_position,difference
1,1,Arsenal FC,€-283.20m,1,0
2,2,Liverpool FC,€-263.40m,8,-6
3,3,Manchester United,€-176.50m,7,-4
4,4,Tottenham Hotspur,€-168.10m,5,-1
5,5,Sunderland AFC,€-136.90m,4,1
6,6,Manchester City,€-136.30m,2,4
7,7,Everton FC,€-117.15m,13,-6
8,8,Nottingham Forest,€-108.00m,19,-11
9,9,Leeds United,€-105.70m,16,-7
10,10,Newcastle United,€-102.85m,14,-4
